In [ ]:
import pandas as pd
import numpy as np

PKL = "/lustre/fsn1/projects/rech/ihj/urb54jd/qtaim_embed_private/data_suba/qm9_with_corrected_qtaim.pkl"

print("Loading...")
df = pd.read_pickle(PKL)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")
print()

# check if split column already exists
if "split" in df.columns:
    print("split column found:")
    print(df["split"].value_counts())
else:
    print("No split column — need to create it")

print()
print("Key columns:")
for c in ["names", "ids", "gdb_num", "split", "homo", "lumo", "gap"]:
    if c in df.columns:
        print(f"  {c:<15}  {type(df[c].iloc[0]).__name__}  sample={df[c].iloc[0]}")
    else:
        print(f"  {c:<15}  MISSING")

print()
print("New corrected columns present?")
new_cols = [c for c in df.columns if c.startswith("new_")]
print(f"  {len(new_cols)} new_* columns: {new_cols[:5]}...")

print()
print("Bond columns:")
for c in ["bonds", "extra_feat_bond_indices_qtaim",
          "extra_feat_bond_Lagrangian_K"]:
    sample = df[c].iloc[0]
    if isinstance(sample, list):
        inner = sample[0] if len(sample)==1 else sample
        print(f"  {c:<45}  list len={len(inner)}")
    else:
        print(f"  {c:<45}  {type(sample).__name__}")

Loading...


In [2]:


# check a few molecules
test_gdbs = [21159, 67265, 78284]

for gdb in test_gdbs:
    row      = df[df["gdb_num"] == gdb].iloc[0]
    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v),max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    raw = row["bonds"]
    if isinstance(raw, list) and len(raw)==1: raw = raw[0]
    bonds_set = {(min(i,j),max(i,j)) for i,j in raw if i!=j}

    missing = len(mg_edges - bonds_set)
    phantom = len(bonds_set - mg_edges)

    print(f"gdb_{gdb:6d}  mg_edges={len(mg_edges):3d}  "
          f"bonds={len(bonds_set):3d}  "
          f"missing={missing}  phantom={phantom}  "
          f"{'✓ correct' if missing==0 and phantom==0 else '✗ still wrong'}")

gdb_ 21159  mg_edges= 15  bonds= 15  missing=0  phantom=0  ✓ correct
gdb_ 67265  mg_edges= 20  bonds= 20  missing=0  phantom=0  ✓ correct
gdb_ 78284  mg_edges= 21  bonds= 21  missing=0  phantom=0  ✓ correct


In [ ]:
import pandas as pd, os

PKL     = "/lustre/fsn1/projects/rech/ihj/urb54jd/qtaim_embed_private/data_suba/qm9_with_corrected_qtaim.pkl"
CSV     = "/lustre/fsn1/projects/rech/ihj/urb54jd/qtaim_embed_private/data_suba/qm9_43k_clean_with_val.csv"
OUT_DIR = "/lustre/fsn1/projects/rech/ihj/urb54jd/qtaim_embed_private/data_suba/filtered_qtaim_fullqm9_corrected"
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading PKL...")
df = pd.read_pickle(PKL)
print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")
print(f"new_* columns: {len([c for c in df.columns if c.startswith('new_')])}")
print(f"split counts:\n{df['split'].value_counts()}")
print()

# quick bond check
for gdb in [21159, 67265, 78284]:
    row      = df[df["gdb_num"] == gdb].iloc[0]
    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v),max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
    raw = row["new_bonds"]
    if isinstance(raw, list) and len(raw)==1: raw = raw[0]
    new_set = {(min(i,j),max(i,j)) for i,j in raw if i!=j}
    print(f"gdb_{gdb}  mg={len(mg_edges)}  new_bonds={len(new_set)}  "
          f"missing={len(mg_edges-new_set)}  phantom={len(new_set-mg_edges)}  "
          f"{'✓' if len(mg_edges-new_set)==0 else '✗'}")

Loading PKL...


In [ ]:
# add val split from csv

print("Loading CSV...")
csv = pd.read_csv(CSV)
print(f"CSV rows: {len(csv)}  columns: {list(csv.columns)}")
print(f"CSV split counts:\n{csv['split'].value_counts()}")
print()

# find the linking column in CSV
for col in csv.columns:
    print(f"  {col:<20}  sample={csv[col].iloc[0]}")


In [ ]:
#merge split from csv and save train/test/val pkl

# merge val split from CSV into df
# (df currently has only train/test from the original PKL)
# CSV has train/val/test with GDB_Index or similar — adjust col name after Cell 2

# ── adjust these after seeing Cell 2 output ───────────────────────────────────
CSV_ID_COL    = "GDB_Index"   # column in CSV that matches gdb_num
CSV_SPLIT_COL = "split"       # column in CSV with train/val/test

# merge
csv_split = csv[[CSV_ID_COL, CSV_SPLIT_COL]].rename(
    columns={CSV_ID_COL: "gdb_num", CSV_SPLIT_COL: "csv_split"})
df = df.merge(csv_split, on="gdb_num", how="left")

print(f"csv_split counts:\n{df['csv_split'].value_counts()}")
print(f"unmatched: {df['csv_split'].isna().sum()}")

# use csv_split as the authoritative split
df["split"] = df["csv_split"]
df = df.drop(columns=["csv_split"])

print(f"\nFinal split counts:\n{df['split'].value_counts()}")

# ── save train / val / test ───────────────────────────────────────────────────
for split_name in ["train", "val", "test"]:
    df_split = df[df["split"] == split_name].reset_index(drop=True)
    out_path = f"{OUT_DIR}/{split_name}_43k.pkl"
    df_split.to_pickle(out_path)
    print(f"Saved {split_name}: {len(df_split)} rows → {out_path}")